# Pilot study (Section 4.1.3): classical ML baselines on GAIA

Mirrors the model family from Section 4.1.1 (sklearn on AIOps Challenge), but applied to the GAIA **diagnosis** task — both FTI (5 classes) and RCL (10 classes).

**Two feature variants:**

| Variant            | Dim   | Honest baseline? | What it tests |
|--------------------|-------|------------------|---------------|
| `full_graph`       | 3840  | yes              | Can simple ML solve diagnosis from flat embeddings? |
| `root_cause_only`  | 384   | **no — diagnostic upper bound** | How much info about the root cause is already in FastText embeddings? |

`root_cause_only` is *not* a fair baseline: the FastText encoder was trained with `__label__{node_idx}{type_idx}` for the root-cause node, so this variant has the root-cause identity leaked into its features. It's reported as the upper bound on what these embeddings can do.

**Six classifiers** — five from Section 4.1.1 (NB, DT, KNN, RF, LR) plus a boosting baseline (LightGBM if available, GradientBoosting otherwise).

**5-fold stratified CV** on `instance` (matches Section 4.2 protocol).

## Setup

In [1]:
from __future__ import annotations
from pathlib import Path
import pickle
import time
import warnings

import numpy as np
import pandas as pd


In [5]:
!pip install scikit-learn
!pip install lightgbm

Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [6]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, ndcg_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except ImportError:
    HAVE_LGBM = False

warnings.filterwarnings('ignore', category=ConvergenceWarning)
print('LightGBM available:' , HAVE_LGBM)


LightGBM available: True


In [11]:
# Paths — adjust DATA_DIR if running outside the local repo layout.
DATA_DIR = Path('../data/gaia')
EMB_DIR = DATA_DIR / 'tmp'
LABELS_CSV = DATA_DIR / 'gaia.csv'
OUT_DIR = Path('../figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
RANDOM_STATE = 42


In [12]:
# Canonical class orders + mapping from raw anomaly_type strings to short labels.
RCL_ORDER = [
    'mobservice1', 'mobservice2',
    'webservice1', 'webservice2',
    'dbservice1', 'dbservice2',
    'logservice1', 'logservice2',
    'redisservice1', 'redisservice2',
]

FTI_SHORT = {
    '[access permission denied exception]': 'access_perm',
    '[file moving program]':                'file_moving',
    '[login failure]':                      'login_failure',
    '[memory_anomalies]':                   'memory_anomalies',
    '[normal memory freed label]':          'normal_mem_freed',
}
FTI_TYPES = ['access_perm', 'file_moving', 'login_failure',
             'memory_anomalies', 'normal_mem_freed']

FTI_HIGHLIGHT = ['access_perm', 'file_moving', 'memory_anomalies', 'normal_mem_freed']
RCL_MINORITY = ['webservice1', 'webservice2', 'dbservice1', 'dbservice2',
                'logservice1', 'logservice2', 'redisservice1', 'redisservice2']


## Load FastText embeddings and labels

In [13]:
with open(EMB_DIR / 'metric.pkl', 'rb') as f:
    metric = np.asarray(pickle.load(f), dtype=np.float32)
with open(EMB_DIR / 'trace.pkl', 'rb') as f:
    trace = np.asarray(pickle.load(f), dtype=np.float32)
with open(EMB_DIR / 'log.pkl', 'rb') as f:
    log = np.asarray(pickle.load(f), dtype=np.float32)

assert metric.shape == trace.shape == log.shape, (
    f'shape mismatch: metric {metric.shape}, trace {trace.shape}, log {log.shape}'
)
metric.shape, trace.shape, log.shape


((16205, 10, 128), (16205, 10, 128), (16205, 10, 128))

In [14]:
labels = pd.read_csv(LABELS_CSV)
labels = (labels[['index', 'anomaly_type', 'instance']]
          .drop_duplicates('index')
          .sort_values('index')
          .reset_index(drop=True))
labels['fti'] = labels['anomaly_type'].map(FTI_SHORT).fillna(labels['anomaly_type'])
assert len(labels) == metric.shape[0], f'{len(labels)} labels vs {metric.shape[0]} embeddings'
labels.head()


,index,anomaly_type,instance,fti
0,0,[memory_anomalies],dbservice1,memory_anomalies
1,1,[normal memory freed label],dbservice1,normal_mem_freed
2,2,[memory_anomalies],dbservice1,memory_anomalies
3,3,[memory_anomalies],dbservice2,memory_anomalies
4,4,[memory_anomalies],dbservice2,memory_anomalies


## Class distribution sanity check

In [15]:
print('FTI distribution:')
print(labels['fti'].value_counts())


FTI distribution:
fti
login_failure       15478
memory_anomalies      652
file_moving            43
normal_mem_freed       17
access_perm            15
Name: count, dtype: int64


In [16]:
print('RCL distribution:')
print(labels['instance'].value_counts().reindex(RCL_ORDER))


RCL distribution:
instance
mobservice1      7816
mobservice2      7796
webservice1        86
webservice2        88
dbservice1         78
dbservice2         96
logservice1        67
logservice2        64
redisservice1      56
redisservice2      58
Name: count, dtype: int64


## Feature variants

`full_graph` flattens all 10 instances × 3 modalities × 128 dim → 3840-dim vector. `root_cause_only` picks only the root-cause node × 3 modalities → 384-dim. The latter has data leakage from FastText labels — see notebook header.

In [17]:
def build_full_graph(metric, trace, log):
    '''(N, 10, 128*3) -> (N, 3840). Honest baseline: no root-cause leakage.'''
    x = np.concatenate([metric, trace, log], axis=2)
    return x.reshape(x.shape[0], -1).astype(np.float32)

def build_root_cause_only(metric, trace, log, labels):
    '''Concatenate only the root-cause node's embedding across modalities.
    Returns (N, 384). DIAGNOSTIC variant — see notebook header.'''
    rc_idx = labels['instance'].map(lambda s: RCL_ORDER.index(s)).values
    rows = np.arange(metric.shape[0])
    return np.concatenate(
        [metric[rows, rc_idx], trace[rows, rc_idx], log[rows, rc_idx]],
        axis=1,
    ).astype(np.float32)


In [18]:
X_full = build_full_graph(metric, trace, log)
X_rc = build_root_cause_only(metric, trace, log, labels)
X_full.shape, X_rc.shape


((16205, 3840), (16205, 384))

## Classifier zoo

Five models from Section 4.1.1 (NB, DT, KNN, RF, LR) + one boosting baseline.
`build_classifiers()` returns fresh instances each call so per-fold fitting is clean.

In [19]:
def build_classifiers() -> dict:
    clfs = {
        'GaussianNB':   GaussianNB(),
        'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=0),
        'KNN':          KNeighborsClassifier(n_neighbors=5),
        'RandomForest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            n_jobs=-1, random_state=0,
        ),
        'LogReg':       LogisticRegression(
            max_iter=2000, class_weight='balanced', n_jobs=-1,
        ),
    }
    if HAVE_LGBM:
        clfs['LightGBM'] = LGBMClassifier(
            n_estimators=200, class_weight='balanced',
            n_jobs=-1, random_state=0, verbose=-1,
        )
    else:
        clfs['GradientBoosting'] = GradientBoostingClassifier(
            n_estimators=200, random_state=0,
        )
    return clfs

list(build_classifiers().keys())


['GaussianNB', 'DecisionTree', 'KNN', 'RandomForest', 'LogReg', 'LightGBM']

## Metric helpers

RCL gets the ranking metrics from the main thesis tables: HR@1, HR@3, NDCG@3. FTI uses macro-F1 and per-class F1.

In [20]:
def hr_at_k(probs: np.ndarray, y_true_idx: np.ndarray, k: int) -> float:
    top_k = np.argsort(-probs, axis=1)[:, :k]
    return float((top_k == y_true_idx[:, None]).any(axis=1).mean())

def ndcg_at_3(probs: np.ndarray, y_true_idx: np.ndarray) -> float:
    n, c = probs.shape
    y_rel = np.zeros((n, c), dtype=np.float32)
    y_rel[np.arange(n), y_true_idx] = 1.0
    return float(ndcg_score(y_rel, probs, k=3))

def align_proba(clf, classes_in_order: list) -> np.ndarray:
    '''Index permutation: clf.classes_ -> classes_in_order.'''
    seen = list(clf.classes_)
    return np.asarray([seen.index(c) if c in seen else -1 for c in classes_in_order])


In [21]:
def evaluate_one_fold(clf, X_tr, X_te, y_tr, y_te, task, classes):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    out = {}
    out['macro_f1'] = f1_score(y_te, y_pred, labels=classes,
                               average='macro', zero_division=0)
    for cls, f1 in zip(classes,
                       f1_score(y_te, y_pred, labels=classes,
                                average=None, zero_division=0)):
        out[f'f1_{cls}'] = float(f1)

    if task == 'RCL' and hasattr(clf, 'predict_proba'):
        raw = clf.predict_proba(X_te)
        perm = align_proba(clf, classes)
        probs = np.zeros((raw.shape[0], len(classes)), dtype=np.float32)
        for j, p in enumerate(perm):
            if p >= 0:
                probs[:, j] = raw[:, p]
        y_te_idx = np.array([classes.index(y) for y in y_te])
        out['hr_1'] = hr_at_k(probs, y_te_idx, 1)
        out['hr_3'] = hr_at_k(probs, y_te_idx, 3)
        out['ndcg_3'] = ndcg_at_3(probs, y_te_idx)
    return out


## 5-fold CV runner

One pass through all folds × both tasks (FTI/RCL) × all classifiers, for a given feature matrix `X`.

In [22]:
def run_cv(X, y_fti, y_rcl, stratify, variant):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    results = {'FTI': {}, 'RCL': {}}
    clf_names = list(build_classifiers().keys())

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X, stratify)):
        print(f'  [{variant}] fold {fold_idx + 1}/{N_FOLDS}'
              f' (train {len(tr_idx)}, test {len(te_idx)})', flush=True)

        scaler = StandardScaler(with_mean=True, with_std=True)
        X_tr = scaler.fit_transform(X[tr_idx])
        X_te = scaler.transform(X[te_idx])

        for task, y, classes in [('FTI', y_fti, FTI_TYPES),
                                 ('RCL', y_rcl, RCL_ORDER)]:
            for clf_name in clf_names:
                clf = build_classifiers()[clf_name]
                t0 = time.time()
                try:
                    m = evaluate_one_fold(clf, X_tr, X_te,
                                          y[tr_idx], y[te_idx],
                                          task, classes)
                    m['_time_sec'] = time.time() - t0
                except Exception as e:
                    print(f'    [{task}/{clf_name}] FAILED:'
                          f' {type(e).__name__}: {e}', flush=True)
                    m = {'_error': f'{type(e).__name__}: {e}'}
                results[task].setdefault(clf_name, []).append(m)
    return results


In [23]:
y_fti = labels['fti'].values
y_rcl = labels['instance'].values
stratify = y_rcl  # matches main TransTVDiag CV protocol


## Run: variant `full_graph` (3840-dim, honest baseline)

In [24]:
print(f'X_full shape: {X_full.shape}')
results_full = run_cv(X_full, y_fti, y_rcl, stratify, variant='full_graph')


X_full shape: (16205, 3840)
  [full_graph] fold 1/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [full_graph] fold 2/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [full_graph] fold 3/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [full_graph] fold 4/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [full_graph] fold 5/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Run: variant `root_cause_only` (384-dim, diagnostic upper bound)

In [25]:
print(f'X_rc shape: {X_rc.shape}')
results_rc = run_cv(X_rc, y_fti, y_rcl, stratify, variant='root_cause_only')


X_rc shape: (16205, 384)
  [root_cause_only] fold 1/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [root_cause_only] fold 2/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [root_cause_only] fold 3/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [root_cause_only] fold 4/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  [root_cause_only] fold 5/5 (train 12964, test 3241)


/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mlcore/conda/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Aggregate per-fold metrics

In [26]:
def summarize(results, variant):
    rows = []
    for task, by_clf in results.items():
        for clf_name, fold_metrics in by_clf.items():
            valid = [m for m in fold_metrics if '_error' not in m]
            if not valid:
                rows.append({'variant': variant, 'task': task,
                             'classifier': clf_name,
                             '_error': 'all folds failed'})
                continue
            row = {'variant': variant, 'task': task,
                   'classifier': clf_name, 'n_folds_ok': len(valid)}
            all_keys = set()
            for m in valid:
                all_keys.update(m.keys())
            for k in sorted(all_keys):
                vals = [m[k] for m in valid
                        if k in m and isinstance(m[k], (int, float))]
                if not vals:
                    continue
                row[f'{k}_mean'] = float(np.mean(vals))
                row[f'{k}_std'] = float(np.std(vals))
            rows.append(row)
    return pd.DataFrame(rows)


In [27]:
df_full = summarize(results_full, 'full_graph')
df_rc = summarize(results_rc, 'root_cause_only')
final = pd.concat([df_full, df_rc], ignore_index=True)
final.shape


(24, 44)

## Headline tables for the thesis

In [28]:
# FTI: macro-F1 + per-class for access_perm and other minority types
fti_view = final[final['task'] == 'FTI'][[
    'variant', 'classifier', 'macro_f1_mean', 'macro_f1_std',
    'f1_access_perm_mean', 'f1_file_moving_mean',
    'f1_memory_anomalies_mean', 'f1_normal_mem_freed_mean',
    'f1_login_failure_mean',
]].copy()
fti_view


,variant,classifier,macro_f1_mean,macro_f1_std,f1_access_perm_mean,f1_file_moving_mean,f1_memory_anomalies_mean,f1_normal_mem_freed_mean,f1_login_failure_mean
0,full_graph,GaussianNB,0.584820,0.072573,0.693333,0.380457,0.854198,0.000000,0.996110
1,full_graph,DecisionTree,0.682520,0.044890,0.711905,0.766183,0.935608,0.000000,0.998902
2,full_graph,KNN,0.625965,0.040677,0.626667,0.546715,0.956733,0.000000,0.999709
3,full_graph,RandomForest,0.755020,0.025339,0.826667,0.971429,0.977102,0.000000,0.999903
4,full_graph,LogReg,0.740165,0.052225,0.871429,0.867326,0.937087,0.026667,0.998318
5,full_graph,LightGBM,0.770752,0.031640,0.904762,0.966667,0.982427,0.000000,0.999903
12,root_cause_only,GaussianNB,0.466690,0.031022,0.384563,0.381400,0.580767,0.000000,0.986720
13,root_cause_only,DecisionTree,0.478400,0.053792,0.271429,0.227079,0.896664,0.000000,0.996828
14,root_cause_only,KNN,0.500631,0.055893,0.311111,0.272381,0.920794,0.000000,0.998868
15,root_cause_only,RandomForest,0.491878,0.091488,0.330000,0.192727,0.938993,0.000000,0.997669


In [29]:
# RCL: macro-F1 + HR@1/HR@3/NDCG@3 (the numbers used in chapter 4.3 tables)
rcl_view = final[final['task'] == 'RCL'][[
    'variant', 'classifier',
    'macro_f1_mean', 'macro_f1_std',
    'hr_1_mean', 'hr_1_std',
    'hr_3_mean', 'hr_3_std',
    'ndcg_3_mean', 'ndcg_3_std',
]].copy()
rcl_view


,variant,classifier,macro_f1_mean,macro_f1_std,hr_1_mean,hr_1_std,hr_3_mean,hr_3_std,ndcg_3_mean,ndcg_3_std
6,full_graph,GaussianNB,0.313245,0.041384,0.861586,0.004529,0.970009,0.002433,0.918495,0.003945
7,full_graph,DecisionTree,0.285125,0.027435,0.865720,0.003811,0.968775,0.002168,0.882629,0.003345
8,full_graph,KNN,0.265519,0.005604,0.841716,0.006609,0.969454,0.000730,0.917116,0.002107
9,full_graph,RandomForest,0.302112,0.017387,0.869732,0.004015,0.970997,0.003048,0.933041,0.002847
10,full_graph,LogReg,0.294757,0.016800,0.861031,0.005046,0.970318,0.002409,0.929264,0.002139
11,full_graph,LightGBM,0.281649,0.027498,0.870410,0.004004,0.970380,0.001421,0.932910,0.001947
18,root_cause_only,GaussianNB,0.990988,0.003779,0.997285,0.000860,0.999506,0.000315,0.998725,0.000374
19,root_cause_only,DecisionTree,0.948146,0.014779,0.994076,0.001399,0.995495,0.001164,0.994764,0.001099
20,root_cause_only,KNN,0.971494,0.018220,0.995989,0.001352,0.997285,0.000924,0.996747,0.001031
21,root_cause_only,RandomForest,0.963737,0.007794,0.995187,0.001318,0.996236,0.000714,0.995719,0.000876


In [34]:
# Minority RCL per-class F1: where the imbalance really bites.
rcl_minority_cols = ['variant', 'classifier'] + [f'f1_{c}_mean' for c in RCL_MINORITY]
rcl_minority_view = final[final['task'] == 'RCL'][rcl_minority_cols].copy()
rcl_minority_view['minority_avg'] = rcl_minority_view[
    [f'f1_{c}_mean' for c in RCL_MINORITY]
].mean(axis=1)
rcl_minority_view


,variant,classifier,f1_webservice1_mean,f1_webservice2_mean,f1_dbservice1_mean,f1_dbservice2_mean,f1_logservice1_mean,f1_logservice2_mean,f1_redisservice1_mean,f1_redisservice2_mean,minority_avg
6,full_graph,GaussianNB,0.257138,0.273968,0.164444,0.168761,0.171795,0.108771,0.065497,0.137969,0.168543
7,full_graph,DecisionTree,0.246665,0.229250,0.126663,0.132548,0.107097,0.099634,0.049333,0.073776,0.133121
8,full_graph,KNN,0.172802,0.160127,0.170180,0.217476,0.074096,0.052439,0.000000,0.070196,0.114664
9,full_graph,RandomForest,0.260340,0.287525,0.228893,0.231257,0.111340,0.064079,0.023529,0.021053,0.153502
10,full_graph,LogReg,0.218611,0.196122,0.160081,0.164744,0.119185,0.107896,0.076028,0.117775,0.145055
11,full_graph,LightGBM,0.217249,0.206992,0.224624,0.165633,0.135082,0.020000,0.018182,0.037053,0.128102
18,root_cause_only,GaussianNB,0.976768,0.977424,0.987097,0.982857,1.000000,1.000000,0.990476,1.000000,0.989328
19,root_cause_only,DecisionTree,0.977423,0.977068,0.663871,0.994595,0.925137,0.975304,0.973781,1.000000,0.935897
20,root_cause_only,KNN,0.987879,0.975000,0.772122,0.994595,1.000000,1.000000,0.990476,1.000000,0.965009
21,root_cause_only,RandomForest,0.993939,0.924534,0.738573,0.994595,1.000000,1.000000,0.990476,1.000000,0.955265


## Human-readable summary

In [30]:
def format_summary(final):
    lines = []
    for variant in ['full_graph', 'root_cause_only']:
        lines.append('')
        lines.append('=' * 72)
        lines.append(f'Variant: {variant}')
        lines.append('=' * 72)
        for task in ['FTI', 'RCL']:
            sub = final[(final['variant'] == variant) & (final['task'] == task)]
            if sub.empty:
                continue
            lines.append('')
            lines.append(f'--- {task} ---')
            for _, row in sub.iterrows():
                line = (f"  {row['classifier']:>18s}: macro-F1 = "
                        f"{row.get('macro_f1_mean', float('nan')):.3f} "
                        f"\u00b1 {row.get('macro_f1_std', float('nan')):.3f}")
                if task == 'RCL' and not pd.isna(row.get('hr_1_mean', np.nan)):
                    line += (
                        f" | HR@1 = {row['hr_1_mean']:.3f}"
                        f" | HR@3 = {row['hr_3_mean']:.3f}"
                        f" | NDCG@3 = {row['ndcg_3_mean']:.3f}"
                    )
                lines.append(line)
    return '\n'.join(lines)

summary_txt = format_summary(final)
print(summary_txt)



Variant: full_graph

--- FTI ---
          GaussianNB: macro-F1 = 0.585 ± 0.073
        DecisionTree: macro-F1 = 0.683 ± 0.045
                 KNN: macro-F1 = 0.626 ± 0.041
        RandomForest: macro-F1 = 0.755 ± 0.025
              LogReg: macro-F1 = 0.740 ± 0.052
            LightGBM: macro-F1 = 0.771 ± 0.032

--- RCL ---
          GaussianNB: macro-F1 = 0.313 ± 0.041 | HR@1 = 0.862 | HR@3 = 0.970 | NDCG@3 = 0.918
        DecisionTree: macro-F1 = 0.285 ± 0.027 | HR@1 = 0.866 | HR@3 = 0.969 | NDCG@3 = 0.883
                 KNN: macro-F1 = 0.266 ± 0.006 | HR@1 = 0.842 | HR@3 = 0.969 | NDCG@3 = 0.917
        RandomForest: macro-F1 = 0.302 ± 0.017 | HR@1 = 0.870 | HR@3 = 0.971 | NDCG@3 = 0.933
              LogReg: macro-F1 = 0.295 ± 0.017 | HR@1 = 0.861 | HR@3 = 0.970 | NDCG@3 = 0.929
            LightGBM: macro-F1 = 0.282 ± 0.027 | HR@1 = 0.870 | HR@3 = 0.970 | NDCG@3 = 0.933

Variant: root_cause_only

--- FTI ---
          GaussianNB: macro-F1 = 0.467 ± 0.031
        DecisionTree:

## Save artifacts

In [31]:
csv_path = OUT_DIR / 'pilot_sklearn_gaia_results.csv'
final.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')


Saved: ../figures/pilot_sklearn_gaia_results.csv


In [32]:
summary_path = OUT_DIR / 'pilot_sklearn_gaia_summary.txt'
summary_path.write_text(summary_txt)
print(f'Saved: {summary_path}')


Saved: ../figures/pilot_sklearn_gaia_summary.txt


In [35]:
# Per-class breakdown for the thesis (access_perm + minority RCL).
fti_pc = fti_view.copy()
fti_pc.insert(0, 'task', 'FTI')
rcl_pc = rcl_minority_view.copy()
rcl_pc.insert(0, 'task', 'RCL')
per_class = pd.concat([fti_pc, rcl_pc], ignore_index=True, sort=False)
per_class_path = OUT_DIR / 'pilot_sklearn_gaia_per_class.csv'
per_class.to_csv(per_class_path, index=False)
print(f'Saved: {per_class_path}')


Saved: ../figures/pilot_sklearn_gaia_per_class.csv


## Next steps for the thesis text

1. Copy `pilot_sklearn_gaia_results.csv` numbers into the table at `thesis/chapter4_experiments.md:89-94`. Extend the table to 7 rows (NB / DT / KNN / RF / LR / GBM / TransTVDiag) and add HR@1, HR@3, NDCG@3 columns for RCL.
2. Add a separate sub-table for the `root_cause_only` variant — label it as a diagnostic upper bound and explain the FastText label leakage.
3. Fill the summary table at `thesis/chapter4_experiments.md:106` with the qualitative outcome (e.g. "FTI: simple models reach ~X macro-F1 vs 0.723 baseline; RCL: minority classes stay at 0 even with class_weight").